[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrychowanda/COMP6577/blob/master/sentiment_emotion_bert.ipynb)

# PhD-Grade Sentiment and Emotion Modeling with BERT-Based Models

**Dataset:** Indonesian sentiment-emotion corpus from Mendeley Data  
**Source:** https://data.mendeley.com/datasets/574v66hf2v

This notebook implements a comprehensive NLP pipeline covering:
1. Dataset loading, EDA, and preprocessing
2. Feature importance analysis (Random Forest, Permutation)
3. Feature selection (Chi-square, Mutual Information, L1)
4. Five BERT-based models (Indonesian + multilingual)
5. Ensemble methods (averaging, max, weighted, majority vote)
6. Full evaluation with confusion matrices, ROC/AUC, classification report
7. Automatic hyperparameter optimisation (Optuna TPE, Grid, Bayesian)
8. Class balancing (SMOTE, oversampling) and text data augmentation (EDA)
9. Explainable AI: SHAP (global) + LIME (local) + attention visualisation
10. Sentiment–Emotion interaction analysis (Chi-square, Cramér's V, LR)


In [ ]:
# ── Install required libraries (run once in Colab) ─────────────────────
!pip install -q transformers datasets accelerate sentencepiece sacremoses
!pip install -q optuna shap lime imbalanced-learn nlpaug
!pip install -q scikit-learn pandas numpy matplotlib seaborn plotly
!pip install -q scipy pingouin statsmodels
!pip install -q scikit-optimize  # for Bayesian search

## 1. Imports and Configuration

In [ ]:
import warnings, os, re, gc, random, math, requests
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from scipy.stats import chi2_contingency

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_curve, auc, average_precision_score,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif, SelectFromModel

from imblearn.over_sampling import SMOTE, RandomOverSampler

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
import lime
import lime.lime_text
from lime.lime_text import LimeTextExplainer

# ── Reproducibility ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

## 2. Dataset Loading

In [ ]:
DATASET_URL = (
    'https://data.mendeley.com/public-files/datasets/574v66hf2v/files/'
    'f258d159-c678-42f1-9634-edf091a0b1f3/file_downloaded'
)

def load_mendeley_dataset(url, local='sentiment_emotion_dataset.csv'):
    if not os.path.exists(local):
        print('Downloading dataset...')
        try:
            r = requests.get(url, timeout=180)
            r.raise_for_status()
        except requests.exceptions.Timeout:
            raise RuntimeError('Download timed out. Try again or download manually.')
        except requests.exceptions.RequestException as err:
            raise RuntimeError(f'Download failed: {err}') from err
        with open(local, 'wb') as fh:
            fh.write(r.content)
        print(f'Saved: {local}')
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(local, encoding=enc)
        except Exception:
            pass
    raise RuntimeError('Could not parse dataset with any encoding.')

df_raw = load_mendeley_dataset(DATASET_URL)
print('Shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
df_raw.head()

In [ ]:
# ── Auto-detect text, sentiment, emotion columns ─────────────────────
def detect_col(df, keywords):
    for col in df.columns:
        if any(k in col.lower() for k in keywords):
            return col
    return None

TEXT_COL      = detect_col(df_raw, ['text','tweet','review','kalimat','konten','content'])
SENTIMENT_COL = detect_col(df_raw, ['sentiment','sentimen','polarity','label'])
EMOTION_COL   = detect_col(df_raw, ['emotion','emosi','feeling'])

cols = df_raw.columns.tolist()
if TEXT_COL is None:      TEXT_COL      = cols[0]
if SENTIMENT_COL is None: SENTIMENT_COL = cols[1] if len(cols) > 1 else cols[0]
if EMOTION_COL is None:   EMOTION_COL   = cols[2] if len(cols) > 2 else None

print(f'Text column      : {TEXT_COL}')
print(f'Sentiment column : {SENTIMENT_COL}')
print(f'Emotion column   : {EMOTION_COL}')
print()
print(df_raw[SENTIMENT_COL].value_counts())

## 3. Preprocessing and EDA

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

df = df_raw.copy().dropna(subset=[TEXT_COL, SENTIMENT_COL]).reset_index(drop=True)
df['text_clean'] = df[TEXT_COL].apply(clean_text)
df = df[df['text_clean'].str.len() > 5].reset_index(drop=True)

le_sent = LabelEncoder()
df['sentiment_label'] = le_sent.fit_transform(df[SENTIMENT_COL].astype(str))
SENTIMENT_CLASSES = le_sent.classes_
N_SENT = len(SENTIMENT_CLASSES)
print(f'Sentiment classes ({N_SENT}): {SENTIMENT_CLASSES}')

if EMOTION_COL and EMOTION_COL in df.columns:
    df = df.dropna(subset=[EMOTION_COL]).reset_index(drop=True)
    le_emo = LabelEncoder()
    df['emotion_label'] = le_emo.fit_transform(df[EMOTION_COL].astype(str))
    EMOTION_CLASSES = le_emo.classes_
else:
    df['emotion_label'] = df['sentiment_label'].copy()
    le_emo = le_sent
    EMOTION_CLASSES = SENTIMENT_CLASSES
    EMOTION_COL = SENTIMENT_COL

N_EMO = len(EMOTION_CLASSES)
print(f'Emotion classes  ({N_EMO}): {EMOTION_CLASSES}')
print(f'Dataset size after cleaning: {len(df)}')

In [ ]:
# ── Class-distribution plots ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, title in [
    (axes[0], SENTIMENT_COL, 'Sentiment Distribution'),
    (axes[1], EMOTION_COL,   'Emotion Distribution'),
]:
    vc = df[col].value_counts()
    ax.bar(vc.index.astype(str), vc.values,
           color=sns.color_palette('Set2', len(vc)))
    for i, v in enumerate(vc.values):
        ax.text(i, v + max(vc)*0.01, str(v), ha='center', fontsize=9)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

# ── Text-length stats ─────────────────────────────────────────────────
df['word_count'] = df['text_clean'].apply(lambda x: len(x.split()))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['word_count'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Word-Count Distribution'); axes[0].set_xlabel('Words')
df.boxplot(column='word_count', by=SENTIMENT_COL, ax=axes[1], patch_artist=True)
axes[1].set_title('Word Count by Sentiment'); axes[1].set_xlabel('Sentiment')
plt.suptitle('')
plt.tight_layout()
plt.savefig('text_statistics.png', bbox_inches='tight')
plt.show()
print(df['word_count'].describe())

## 4. Feature Importance Analysis

In [ ]:
# ── TF-IDF (5000 features, bigrams) ─────────────────────────────────
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                        sublinear_tf=True, min_df=2)
X_tfidf = tfidf.fit_transform(df['text_clean'])
feat_names = tfidf.get_feature_names_out()
y_sent = df['sentiment_label'].values
y_emo  = df['emotion_label'].values

# ── Random Forest importance ──────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_tfidf, y_sent)
importances = rf.feature_importances_
top_k = 25
top_idx = np.argsort(importances)[::-1][:top_k]

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(feat_names[top_idx][::-1], importances[top_idx][::-1],
        color=sns.color_palette('viridis', top_k))
ax.set_xlabel('Gini Importance')
ax.set_title(f'Top-{top_k} TF-IDF Feature Importances (Random Forest)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

# ── Permutation importance ────────────────────────────────────────────
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tfidf, y_sent, test_size=0.2, random_state=SEED, stratify=y_sent)
rf_pi = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_pi.fit(X_tr, y_tr)
pi = permutation_importance(rf_pi, X_va, y_va, n_repeats=5,
                             random_state=SEED, n_jobs=-1)
pi_idx = pi.importances_mean.argsort()[::-1][:20]

fig, ax = plt.subplots(figsize=(12, 6))
ax.boxplot(pi.importances[pi_idx].T, vert=False,
           labels=feat_names[pi_idx])
ax.set_xlabel('Permutation Importance')
ax.set_title('Top-20 Permutation Feature Importances', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('permutation_importance.png', bbox_inches='tight')
plt.show()

## 5. Feature Selection

In [ ]:
# ── Chi-square selection ─────────────────────────────────────────────
sel_chi2 = SelectKBest(chi2, k=1000).fit(X_tfidf, y_sent)
chi2_scores = sel_chi2.scores_
chi2_top = np.argsort(chi2_scores)[::-1][:20]

# ── Mutual Information selection ──────────────────────────────────────
sel_mi = SelectKBest(mutual_info_classif, k=1000).fit(X_tfidf, y_sent)
mi_scores = sel_mi.scores_
mi_top = np.argsort(mi_scores)[::-1][:20]

# ── L1 (Lasso) selection ──────────────────────────────────────────────
lsvc = LinearSVC(C=0.05, penalty='l1', dual=False, max_iter=2000,
                 random_state=SEED)
sel_l1 = SelectFromModel(lsvc).fit(X_tfidf, y_sent)
n_l1 = sel_l1.transform(X_tfidf).shape[1]

print(f'Features selected — Chi2 : 1000 (from {X_tfidf.shape[1]})')
print(f'Features selected — MI   : 1000')
print(f'Features selected — L1   : {n_l1}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(feat_names[chi2_top][::-1], chi2_scores[chi2_top][::-1], color='coral')
axes[0].set_title('Top-20 Chi-Square Scores', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Chi2 Score')

axes[1].barh(feat_names[mi_top][::-1], mi_scores[mi_top][::-1], color='mediumseagreen')
axes[1].set_title('Top-20 Mutual Information Scores', fontsize=12, fontweight='bold')
axes[1].set_xlabel('MI Score')

plt.tight_layout()
plt.savefig('feature_selection_scores.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── t-SNE visualisation of TF-IDF feature space ──────────────────────
svd = TruncatedSVD(n_components=50, random_state=SEED)
X_svd = svd.fit_transform(X_tfidf)

n_sample = min(2000, len(df))
idx_s = np.random.choice(len(df), n_sample, replace=False)

# n_iter=500 is sufficient for visualisation; increase to 1000+ for higher quality
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=500)
X_2d = tsne.fit_transform(X_svd[idx_s])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, labs, classes, title in [
    (axes[0], y_sent[idx_s], SENTIMENT_CLASSES, 'Sentiment'),
    (axes[1], y_emo[idx_s],  EMOTION_CLASSES,   'Emotion'),
]:
    pal = sns.color_palette('tab10', len(classes))
    for cid, cname in enumerate(classes):
        m = labs == cid
        ax.scatter(X_2d[m,0], X_2d[m,1], c=[pal[cid]],
                   label=cname, alpha=0.6, s=15)
    ax.legend(fontsize=9, markerscale=2)
    ax.set_title(f't-SNE: {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig('tsne_visualization.png', bbox_inches='tight')
plt.show()

## 6. Data Split and PyTorch Dataset

In [ ]:
texts       = df['text_clean'].tolist()
sent_labels = df['sentiment_label'].tolist()
emo_labels  = df['emotion_label'].tolist()

X_train, X_tmp, ys_train, ys_tmp, ye_train, ye_tmp = train_test_split(
    texts, sent_labels, emo_labels,
    test_size=0.30, random_state=SEED, stratify=sent_labels)

X_val, X_test, ys_val, ys_test, ye_val, ye_test = train_test_split(
    X_tmp, ys_tmp, ye_tmp,
    test_size=0.50, random_state=SEED, stratify=ys_tmp)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')


class SentEmoDataset(Dataset):
    def __init__(self, texts, sent_labels, emo_labels, tokenizer, max_len=128):
        self.texts, self.sent_labels = texts, sent_labels
        self.emo_labels, self.tokenizer = emo_labels, tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt')
        tti = enc.get('token_type_ids',
                      torch.zeros(self.max_len, dtype=torch.long))
        return {
            'input_ids':       enc['input_ids'].squeeze(0),
            'attention_mask':  enc['attention_mask'].squeeze(0),
            'token_type_ids':  tti.squeeze(0),
            'sentiment_label': torch.tensor(self.sent_labels[idx], dtype=torch.long),
            'emotion_label':   torch.tensor(self.emo_labels[idx],  dtype=torch.long),
        }

## 7. Training and Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        tti   = batch['token_type_ids'].to(device)
        labs  = batch['sentiment_label'].to(device)
        optimizer.zero_grad()
        tti_arg = tti if tti.sum() > 0 else None
        out = model(input_ids=ids, attention_mask=mask,
                    token_type_ids=tti_arg, labels=labs)
        out.loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
        preds = out.logits.argmax(-1)
        correct += (preds == labs).sum().item()
        total   += labs.size(0)
    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate_loader(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labs, all_probs = 0.0, [], [], []
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        tti  = batch['token_type_ids'].to(device)
        labs = batch['sentiment_label'].to(device)
        tti_arg = tti if tti.sum() > 0 else None
        out = model(input_ids=ids, attention_mask=mask,
                    token_type_ids=tti_arg, labels=labs)
        total_loss += out.loss.item()
        probs = F.softmax(out.logits, -1)
        preds = probs.argmax(-1)
        all_preds.extend(preds.cpu().numpy())
        all_labs.extend(labs.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    acc  = accuracy_score(all_labs, all_preds)
    f1   = f1_score(all_labs, all_preds, average='weighted', zero_division=0)
    prec = precision_score(all_labs, all_preds, average='weighted', zero_division=0)
    rec  = recall_score(all_labs, all_preds, average='weighted', zero_division=0)
    return dict(loss=total_loss/len(loader), acc=acc, f1=f1,
                precision=prec, recall=rec,
                preds=all_preds, labels=all_labs, probs=all_probs)


def train_bert(model_name, n_classes,
               tr_texts, tr_sent, tr_emo,
               va_texts, va_sent, va_emo,
               epochs=3, batch_size=16, lr=2e-5, max_len=128):
    print(f'\n===== {model_name} =====')
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=n_classes, ignore_mismatched_sizes=True).to(DEVICE)

    tr_ds = SentEmoDataset(tr_texts, tr_sent, tr_emo, tok, max_len)
    va_ds = SentEmoDataset(va_texts, va_sent, va_emo, tok, max_len)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=2)
    va_dl = DataLoader(va_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(tr_dl) * epochs
    sch = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.1*total), num_training_steps=total)

    hist = dict(train_loss=[], train_acc=[], val_loss=[], val_acc=[], val_f1=[])
    best_f1, best_st = 0.0, None
    for ep in range(1, epochs+1):
        tl, ta = train_one_epoch(model, tr_dl, opt, sch, DEVICE)
        vm = evaluate_loader(model, va_dl, DEVICE)
        hist['train_loss'].append(tl); hist['train_acc'].append(ta)
        hist['val_loss'].append(vm['loss'])
        hist['val_acc'].append(vm['acc'])
        hist['val_f1'].append(vm['f1'])
        print(f'  Ep {ep}/{epochs}  tl={tl:.4f} ta={ta:.4f}'
              f'  vl={vm["loss"]:.4f} vf1={vm["f1"]:.4f}')
        if vm['f1'] > best_f1:
            best_f1 = vm['f1']
            best_st = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_st)
    return model, tok, hist, vm

## 8. Training Five BERT-Based Models

In [ ]:
MODEL_CONFIGS = [
    {'name': 'indobenchmark/indobert-base-p1',      'short': 'IndoBERT'},
    {'name': 'indobenchmark/indobert-lite-base-p1', 'short': 'IndoBERT-Lite'},
    {'name': 'indolem/indobertweet-base-uncased',    'short': 'IndoBERTweet'},
    {'name': 'bert-base-multilingual-cased',         'short': 'mBERT'},
    {'name': 'xlm-roberta-base',                     'short': 'XLM-RoBERTa'},
]

EPOCHS, BATCH_SIZE, MAX_LEN, LR = 3, 16, 128, 2e-5

trained_models  = {}   # short -> (model, tokenizer)
model_histories = {}   # short -> history
model_val_mets  = {}   # short -> val metrics (last epoch)

for cfg in MODEL_CONFIGS:
    m, tok, hist, vm = train_bert(
        cfg['name'], N_SENT,
        X_train, ys_train, ye_train,
        X_val,   ys_val,   ye_val,
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, max_len=MAX_LEN)
    trained_models[cfg['short']]  = (m, tok)
    model_histories[cfg['short']] = hist
    model_val_mets[cfg['short']]  = vm
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nAll 5 models trained.')

In [ ]:
# ── Training curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, (short, hist) in enumerate(model_histories.items()):
    ax = axes[i]
    ep = range(1, len(hist['train_loss'])+1)
    ax.plot(ep, hist['train_loss'], 'b-o', label='Train Loss')
    ax.plot(ep, hist['val_loss'],   'r-s', label='Val Loss')
    ax2 = ax.twinx()
    ax2.plot(ep, hist['val_f1'], 'g--^', label='Val F1')
    ax.set_title(short, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax2.set_ylabel('F1', color='green')
    lines = ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
    lbls  = ax.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
    ax.legend(lines, lbls, fontsize=8)
for j in range(len(model_histories), len(axes)): axes[j].set_visible(False)
plt.suptitle('Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', bbox_inches='tight')
plt.show()

## 9. Model Evaluation

In [ ]:
@torch.no_grad()
def test_metrics(model, tok, texts, s_labs, e_labs):
    ds = SentEmoDataset(texts, s_labs, e_labs, tok, MAX_LEN)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return evaluate_loader(model, dl, DEVICE)

test_results = {}
for short, (m, tok) in trained_models.items():
    mets = test_metrics(m, tok, X_test, ys_test, ye_test)
    test_results[short] = mets
    print(f'{short:20s}  Acc={mets["acc"]:.4f}  F1={mets["f1"]:.4f}  '
          f'Prec={mets["precision"]:.4f}  Rec={mets["recall"]:.4f}')

In [ ]:
metrics_df = pd.DataFrame([
    {'Model': s, 'Accuracy': round(m['acc'],4),
     'F1 (Weighted)': round(m['f1'],4),
     'Precision': round(m['precision'],4),
     'Recall': round(m['recall'],4)}
    for s, m in test_results.items()
]).sort_values('F1 (Weighted)', ascending=False).reset_index(drop=True)
display(metrics_df)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(metrics_df))
w = 0.2
met_cols = ['Accuracy','F1 (Weighted)','Precision','Recall']
colors   = ['#4C72B0','#DD8452','#55A868','#C44E52']
for i, (mc, col) in enumerate(zip(met_cols, colors)):
    ax.bar(x + i*w, metrics_df[mc], w, label=mc, color=col, alpha=0.85)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(metrics_df['Model'], rotation=15, ha='right')
ax.set_ylim(0, 1.15)
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
n_mod = len(test_results)
n_cols = 3
n_rows = math.ceil(n_mod / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
axes = np.array(axes).flatten()
for i, (short, m) in enumerate(test_results.items()):
    cm = confusion_matrix(m['labels'], m['preds'])
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=SENTIMENT_CLASSES,
                yticklabels=SENTIMENT_CLASSES, ax=axes[i])
    axes[i].set_title(f'{short}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')
for j in range(n_mod, len(axes)): axes[j].set_visible(False)
plt.suptitle('Normalised Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(test_results),
                          figsize=(6*len(test_results), 5))
if len(test_results) == 1: axes = [axes]
for ax, (short, m) in zip(axes, test_results.items()):
    y_bin = label_binarize(m['labels'], classes=list(range(N_SENT)))
    y_prob = np.array(m['probs'])
    for cid in range(N_SENT):
        fpr, tpr, _ = roc_curve(y_bin[:, cid], y_prob[:, cid])
        ax.plot(fpr, tpr, label=f'{SENTIMENT_CLASSES[cid]} (AUC={auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--')
    ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC — {short}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

## 10. Ensemble Methods

In [ ]:
@torch.no_grad()
def get_probs(model, tok, texts, batch_size=32):
    dummy = [0]*len(texts)
    ds = SentEmoDataset(texts, dummy, dummy, tok, MAX_LEN)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    probs = []
    for batch in dl:
        out = model(input_ids=batch['input_ids'].to(DEVICE),
                    attention_mask=batch['attention_mask'].to(DEVICE))
        probs.extend(F.softmax(out.logits, -1).cpu().numpy())
    return np.array(probs)

all_test_probs = []
for short, (m, tok) in trained_models.items():
    p = get_probs(m, tok, X_test)
    all_test_probs.append(p)
    print(f'{short}: {p.shape}')
all_test_probs = np.stack(all_test_probs)  # (5, N_test, N_classes)


def ensemble_eval(probs_stack, true_labels, strategy='avg', weights=None):
    from scipy.stats import mode
    if strategy == 'avg':
        combined = probs_stack.mean(0)
    elif strategy == 'max':
        combined = probs_stack.max(0)
    elif strategy == 'weighted':
        w = np.array(weights)[:,None,None]
        combined = (probs_stack * w).sum(0) / w.sum()
    elif strategy == 'majority':
        votes = probs_stack.argmax(-1)   # (5, N)
        preds = mode(votes, axis=0).mode.flatten()
        return accuracy_score(true_labels, preds), \
               f1_score(true_labels, preds, average='weighted', zero_division=0), preds
    else:
        combined = probs_stack.mean(0)
    preds = combined.argmax(-1)
    return (accuracy_score(true_labels, preds),
            f1_score(true_labels, preds, average='weighted', zero_division=0), preds)

f1_weights = [model_val_mets[s]['f1'] for s in trained_models]

ensemble_results = {}
for strat, kw in [
    ('Average Probability', dict(strategy='avg')),
    ('Max Probability',     dict(strategy='max')),
    ('Weighted (val-F1)',   dict(strategy='weighted', weights=f1_weights)),
    ('Majority Vote',       dict(strategy='majority')),
]:
    acc, f1, preds = ensemble_eval(all_test_probs, ys_test, **kw)
    ensemble_results[strat] = dict(acc=acc, f1=f1, preds=preds)
    print(f'{strat:30s}  Acc={acc:.4f}  F1={f1:.4f}')

In [ ]:
all_entries = (
    [{'Model': k, 'F1': v['f1'], 'Type': 'Individual'} for k,v in test_results.items()] +
    [{'Model': k, 'F1': v['f1'], 'Type': 'Ensemble'}   for k,v in ensemble_results.items()]
)
all_df = pd.DataFrame(all_entries)

fig, ax = plt.subplots(figsize=(14, 5))
palette = {'Individual': '#4C72B0', 'Ensemble': '#DD8452'}
for t, grp in all_df.groupby('Type'):
    ax.bar(grp['Model'], grp['F1'], label=t, color=palette[t], alpha=0.85)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Weighted F1')
ax.set_title('Individual vs Ensemble F1', fontsize=14, fontweight='bold')
ax.set_xticklabels(all_df['Model'].tolist(), rotation=20, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('ensemble_comparison.png', bbox_inches='tight')
plt.show()

## 11. Automatic Hyperparameter Optimisation

In [ ]:
# ── Optuna TPE ────────────────────────────────────────────────────────
HPO_MODEL = 'indobenchmark/indobert-base-p1'
HPO_EPOCHS = 2

def objective(trial):
    lr    = trial.suggest_float('lr', 1e-5, 5e-5, log=True)
    bs    = trial.suggest_categorical('batch_size', [8, 16, 32])
    ml    = trial.suggest_categorical('max_len', [64, 128])
    wd    = trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True)
    _, _, _, vm = train_bert(
        HPO_MODEL, N_SENT,
        X_train, ys_train, ye_train,
        X_val,   ys_val,   ye_val,
        epochs=HPO_EPOCHS, batch_size=bs, lr=lr, max_len=ml)
    return vm['f1']

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
# n_trials=10 is a quick demo; increase to 30-50 for production HPO
study.optimize(objective, n_trials=10, show_progress_bar=True)
print('Best Optuna params:', study.best_params)
print('Best Optuna F1:    ', round(study.best_value, 4))

In [ ]:
# ── Optuna visualisation ─────────────────────────────────────────────
try:
    from optuna.visualization.matplotlib import (
        plot_optimization_history, plot_param_importances)
    plot_optimization_history(study)
    plt.title('Optuna: Optimisation History')
    plt.tight_layout()
    plt.savefig('optuna_history.png', bbox_inches='tight')
    plt.show()
    plot_param_importances(study)
    plt.title('Optuna: Hyperparameter Importance')
    plt.tight_layout()
    plt.savefig('optuna_param_importance.png', bbox_inches='tight')
    plt.show()
except Exception as e:
    print('Optuna viz error:', e)

# ── Grid Search on LR + Logistic Regression (lightweight demo) ───────
param_grid = {'C': [0.01, 0.1, 1, 10], 'max_iter': [500, 1000]}
gs = GridSearchCV(LogisticRegression(multi_class='auto', random_state=SEED),
                  param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
gs.fit(X_tfidf[:len(X_train)], y_sent[:len(X_train)])
print('Grid Search best params:', gs.best_params_)
print('Grid Search best F1:    ', round(gs.best_score_, 4))

# ── Bayesian Optimisation ─────────────────────────────────────────────
try:
    from skopt import BayesSearchCV
    from skopt.space import Real, Integer
    bs_cv = BayesSearchCV(
        LogisticRegression(multi_class='auto', random_state=SEED),
        {'C': Real(1e-3, 1e2, prior='log-uniform'), 'max_iter': Integer(200,2000)},
        n_iter=15, cv=3, scoring='f1_weighted', n_jobs=-1, random_state=SEED)
    bs_cv.fit(X_tfidf[:len(X_train)], y_sent[:len(X_train)])
    print('Bayesian best params:', bs_cv.best_params_)
    print('Bayesian best F1:    ', round(bs_cv.best_score_, 4))
except ImportError:
    print('scikit-optimize not available; skipping Bayesian Search.')

In [ ]:
trials_df = study.trials_dataframe()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(trials_df['number'], trials_df['value'], 'bo-', ms=6)
axes[0].axhline(study.best_value, color='red', ls='--',
                label=f'Best={study.best_value:.4f}')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val F1')
axes[0].set_title('Optuna Trial History', fontsize=13, fontweight='bold')
axes[0].legend()

lr_v = [t.params.get('lr', np.nan) for t in study.trials]
f1_v = [t.value for t in study.trials]
axes[1].scatter(lr_v, f1_v, c='royalblue', s=80, alpha=0.8)
axes[1].set_xscale('log')
axes[1].set_xlabel('Learning Rate (log)')
axes[1].set_ylabel('Val F1')
axes[1].set_title('LR vs Val F1', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('hpo_results.png', bbox_inches='tight')
plt.show()

## 12. Class Balancing and Data Augmentation

In [ ]:
from collections import Counter
print('Train distribution (original):', Counter(ys_train))

X_tr_arr = X_tfidf[:len(X_train)].toarray()

# ── SMOTE ─────────────────────────────────────────────────────────────
try:
    k = min(5, min(Counter(ys_train).values()) - 1)
    smote = SMOTE(random_state=SEED, k_neighbors=max(1, k))
    X_sm, y_sm = smote.fit_resample(X_tr_arr, ys_train)
    print('After SMOTE:', Counter(y_sm))
except Exception as e:
    print('SMOTE error:', e)
    X_sm, y_sm = X_tr_arr, np.array(ys_train)

# ── Random Oversampling ────────────────────────────────────────────────
ros = RandomOverSampler(random_state=SEED)
X_ros, y_ros = ros.fit_resample(X_tr_arr, ys_train)
print('After RandomOverSampler:', Counter(y_ros))

# ── Visualise ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (cnt, title) in zip(axes, [
    (Counter(ys_train), 'Original'),
    (Counter(y_sm),     'After SMOTE'),
    (Counter(y_ros),    'After Oversampling'),
]):
    ks = sorted(cnt.keys())
    ax.bar([SENTIMENT_CLASSES[k] for k in ks], [cnt[k] for k in ks],
           color=sns.color_palette('pastel', len(ks)))
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('class_balancing.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Easy Data Augmentation (EDA) ─────────────────────────────────────
def eda_augment(text, alpha=0.1, n_aug=1):
    words = text.split()
    if len(words) < 3: return [text] * n_aug
    n_ops = max(1, int(alpha * len(words)))
    results = []
    for _ in range(n_aug):
        w = words.copy()
        # random deletion
        w = [x for x in w if random.random() > alpha] or [random.choice(words)]
        # random swap
        for _ in range(n_ops):
            if len(w) >= 2:
                i, j = random.sample(range(len(w)), 2)
                w[i], w[j] = w[j], w[i]
        results.append(' '.join(w))
    return results

cnt = Counter(ys_train)
max_c = max(cnt.values())
aug_texts  = list(X_train)
aug_slabs  = list(ys_train)
aug_elabs  = list(ye_train)

for cid, c_cnt in cnt.items():
    if c_cnt < max_c:
        src_t = [t for t, l in zip(X_train, ys_train) if l == cid]
        src_e = [e for e, l in zip(ye_train, ys_train) if l == cid]
        need  = max_c - c_cnt
        added = 0
        while added < need:
            idx = random.randint(0, len(src_t)-1)
            at = eda_augment(src_t[idx], n_aug=1)[0]
            aug_texts.append(at)
            aug_slabs.append(cid)
            aug_elabs.append(src_e[idx])
            added += 1

print(f'Before augmentation: {len(X_train)}')
print(f'After  augmentation: {len(aug_texts)}')
print('Augmented distribution:', Counter(aug_slabs))

In [ ]:
# ── Re-train best model with augmented data and best HPO params ──────
best_short = max(model_val_mets, key=lambda k: model_val_mets[k]['f1'])
best_name  = next(c['name'] for c in MODEL_CONFIGS if c['short'] == best_short)
print(f'Best model: {best_short} ({best_name})')

bp = study.best_params
aug_model, aug_tok, aug_hist, aug_vm = train_bert(
    best_name, N_SENT,
    aug_texts, aug_slabs, aug_elabs,
    X_val, ys_val, ye_val,
    epochs=EPOCHS,
    batch_size=bp.get('batch_size', BATCH_SIZE),
    lr=bp.get('lr', LR),
    max_len=bp.get('max_len', MAX_LEN)
)
aug_test_m = test_metrics(aug_model, aug_tok, X_test, ys_test, ye_test)

print(f'\nOriginal best test F1 : {test_results[best_short]["f1"]:.4f}')
print(f'Augmented+HPO test F1 : {aug_test_m["f1"]:.4f}')
print(f'Improvement           : {aug_test_m["f1"] - test_results[best_short]["f1"]:+.4f}')

## 13. Explainable AI (XAI) — SHAP

In [ ]:
# ── SHAP on TF-IDF + Logistic Regression ─────────────────────────────
lr_model = LogisticRegression(max_iter=2000, multi_class='auto', random_state=SEED)
lr_model.fit(X_tfidf[:len(X_train)], y_sent[:len(X_train)])

shap_bg_n = min(100, len(X_train))
shap_bg   = shap.sample(X_tfidf[:len(X_train)], shap_bg_n, random_state=SEED)

explainer  = shap.KernelExplainer(lr_model.predict_proba, shap_bg)
shap_n     = min(50, len(X_test))
shap_test_X = X_tfidf[len(X_train)+len(X_val): len(X_train)+len(X_val)+shap_n]
# nsamples=100 balances accuracy and speed; increase for more precise estimates
shap_vals  = explainer.shap_values(shap_test_X, nsamples=100)
print('SHAP values shape:', np.array(shap_vals).shape)

In [ ]:
shap_arr = shap_vals[0] if isinstance(shap_vals, list) else shap_vals

# Summary plot (dot)
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_arr, shap_test_X.toarray(),
                  feature_names=feat_names, max_display=20, show=False)
plt.title('SHAP Summary Plot (dot)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', bbox_inches='tight')
plt.show()

# Bar plot (global importance)
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_arr, shap_test_X.toarray(),
                  feature_names=feat_names, plot_type='bar',
                  max_display=20, show=False)
plt.title('SHAP Feature Importance (bar)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', bbox_inches='tight')
plt.show()

# Waterfall plot for one sample
try:
    expl_obj = shap.Explanation(
        values=shap_arr[0],
        base_values=explainer.expected_value[0],
        data=shap_test_X[0].toarray().flatten(),
        feature_names=feat_names
    )
    plt.figure(figsize=(14, 6))
    shap.waterfall_plot(expl_obj, max_display=15, show=False)
    plt.title('SHAP Waterfall — Sample 0', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_waterfall.png', bbox_inches='tight')
    plt.show()
except Exception as e:
    print('Waterfall plot error:', e)

## 14. Explainable AI (XAI) — LIME

In [ ]:
@torch.no_grad()
def bert_pred(texts):
    enc = aug_tok(
        texts, padding=True, truncation=True,
        max_length=MAX_LEN, return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    aug_model.eval()
    return F.softmax(aug_model(**enc).logits, -1).cpu().numpy()

lime_exp = LimeTextExplainer(
    class_names=SENTIMENT_CLASSES.tolist(), random_state=SEED)

lime_results = []
for i in range(min(5, len(X_test))):
    exp = lime_exp.explain_instance(
        X_test[i], bert_pred, num_features=10, num_samples=100)
    lime_results.append(exp)
    print(f'Sample {i}: true={SENTIMENT_CLASSES[ys_test[i]]}  '
          f'text="{X_test[i][:60]}..."')

print('\nLIME explanations computed.')

In [ ]:
fig, axes = plt.subplots(1, min(3, len(lime_results)), figsize=(18, 6))
if len(lime_results) == 1: axes = [axes]
for i, (ax, exp) in enumerate(zip(axes, lime_results[:3])):
    ww = exp.as_list()
    ws = [w for w,_ in ww]; vs = [v for _,v in ww]
    cols = ['#2ecc71' if v >= 0 else '#e74c3c' for v in vs]
    ax.barh(ws[::-1], vs[::-1], color=cols[::-1])
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('LIME weight')
    ax.set_title(f'Sample {i+1} — True: {SENTIMENT_CLASSES[ys_test[i]]}',
                 fontsize=10, fontweight='bold')
plt.suptitle('LIME Local Explanations (BERT)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('lime_explanations.png', bbox_inches='tight')
plt.show()

## 15. BERT Attention Visualisation

In [ ]:
@torch.no_grad()
def get_attn(text, tok, model, layer=-1):
    inp = tok(text, return_tensors='pt',
              truncation=True, max_length=MAX_LEN).to(DEVICE)
    out = model(**inp, output_attentions=True)
    attn = out.attentions[layer][0].mean(0).cpu().numpy()
    tokens = tok.convert_ids_to_tokens(inp['input_ids'][0])
    return tokens, attn

tokens, attn = get_attn(X_test[0], aug_tok, aug_model)
max_t = min(20, len(tokens))

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(attn[:max_t, :max_t], cmap='YlOrRd')
ax.set_xticks(range(max_t)); ax.set_yticks(range(max_t))
ax.set_xticklabels(tokens[:max_t], rotation=90, fontsize=9)
ax.set_yticklabels(tokens[:max_t], fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('BERT Attention (last layer, avg heads)\n'
             f'"{X_test[0][:60]}..."',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('bert_attention.png', bbox_inches='tight')
plt.show()

## 16. Sentiment–Emotion Interaction Analysis

In [ ]:
# ── Crosstabulation ───────────────────────────────────────────────────
ct = pd.crosstab(df[SENTIMENT_COL].astype(str), df[EMOTION_COL].astype(str))
display(ct)

# ── Chi-square test of independence ──────────────────────────────────
chi2_stat, p_val, dof, expected = chi2_contingency(ct)
n = ct.sum().sum()
cramers_v = np.sqrt(chi2_stat / (n * (min(ct.shape)-1)))
print(f'Chi2={chi2_stat:.4f}  df={dof}  p={p_val:.4e}')
print(f"Cramér's V = {cramers_v:.4f}")
if p_val < 0.05:
    print('→ Significant association between sentiment and emotion (p < 0.05)')
else:
    print('→ No significant association (p ≥ 0.05)')

In [ ]:
# ── Heatmap ──────────────────────────────────────────────────────────
ct_norm = ct.div(ct.sum(axis=1), axis=0)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(ct,      annot=True, fmt='d',   cmap='Blues',  ax=axes[0])
axes[0].set_title('Sentiment × Emotion (Counts)', fontsize=12, fontweight='bold')
sns.heatmap(ct_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Sentiment → Emotion (Row-normalised)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_emotion_heatmap.png', bbox_inches='tight')
plt.show()

# ── Sankey diagram (Plotly) ───────────────────────────────────────────
sent_u = sorted(ct.index.tolist())
emo_u  = sorted(ct.columns.tolist())
all_nl = sent_u + emo_u
melt   = ct.reset_index().melt(id_vars=SENTIMENT_COL,
                                var_name=EMOTION_COL, value_name='cnt')
srcs, tgts, vals = [], [], []
for _, row in melt.iterrows():
    srcs.append(all_nl.index(str(row[SENTIMENT_COL])))
    tgts.append(all_nl.index(str(row[EMOTION_COL])))
    vals.append(int(row['cnt']))

fig_sk = go.Figure(go.Sankey(
    node=dict(label=all_nl, pad=15, thickness=20),
    link=dict(source=srcs, target=tgts, value=vals)
))
fig_sk.update_layout(title_text='Sankey: Sentiment → Emotion',
                      font_size=12, height=450)
fig_sk.write_html('sankey_sentiment_emotion.html')
fig_sk.show()

In [ ]:
# ── Logistic regression: does sentiment predict emotion? ─────────────
lr_s2e = LogisticRegression(multi_class='multinomial', max_iter=2000,
                              random_state=SEED)
lr_s2e.fit(df['sentiment_label'].values.reshape(-1,1),
           df['emotion_label'].values)
pe = lr_s2e.predict(df['sentiment_label'].values.reshape(-1,1))
print(f'Sentiment → Emotion  F1={f1_score(df["emotion_label"],pe,average="weighted",zero_division=0):.4f}')

lr_e2s = LogisticRegression(multi_class='multinomial', max_iter=2000,
                              random_state=SEED)
lr_e2s.fit(df['emotion_label'].values.reshape(-1,1),
           df['sentiment_label'].values)
ps = lr_e2s.predict(df['emotion_label'].values.reshape(-1,1))
print(f'Emotion   → Sentiment F1={f1_score(df["sentiment_label"],ps,average="weighted",zero_division=0):.4f}')

print(f"\nCramér's V = {cramers_v:.4f}")
strength = ('strong' if cramers_v>0.5 else
            'moderate' if cramers_v>0.3 else
            'weak' if cramers_v>0.1 else 'very weak')
print(f'Association strength: {strength}')

## 17. Final Leaderboard and Radar Chart

In [ ]:
aug_entry = {
    'Model':          f'{best_short} (Aug+HPO)',
    'Accuracy':       round(aug_test_m['acc'], 4),
    'F1 (Weighted)':  round(aug_test_m['f1'], 4),
    'Precision':      round(aug_test_m['precision'], 4),
    'Recall':         round(aug_test_m['recall'], 4),
}
final_df = pd.concat(
    [metrics_df, pd.DataFrame([aug_entry])], ignore_index=True
).sort_values('F1 (Weighted)', ascending=False).reset_index(drop=True)
print('=== Final Leaderboard ===')
display(final_df)

# ── Radar chart ───────────────────────────────────────────────────────
cats   = ['Accuracy','F1','Precision','Recall']
N      = len(cats)
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
pal = sns.color_palette('tab10', len(final_df))
for i, row in final_df.iterrows():
    vals = [row['Accuracy'], row['F1 (Weighted)'],
            row['Precision'], row['Recall'], row['Accuracy']]
    ax.plot(angles, vals, 'o-', lw=2, color=pal[i], label=row['Model'])
    ax.fill(angles, vals, alpha=0.05, color=pal[i])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, size=12)
ax.set_ylim(0, 1)
ax.set_title('Radar Chart: Model Performance', size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig('radar_chart.png', bbox_inches='tight')
plt.show()

In [ ]:
print('===== Detailed Classification Reports =====')
for short, m in test_results.items():
    print(f'\n--- {short} ---')
    print(classification_report(m['labels'], m['preds'],
                                 target_names=SENTIMENT_CLASSES, zero_division=0))
print('\n--- Augmented + HPO ---')
print(classification_report(aug_test_m['labels'], aug_test_m['preds'],
                              target_names=SENTIMENT_CLASSES, zero_division=0))

## 18. Save Artifacts

In [ ]:
aug_model.save_pretrained('best_model')
aug_tok.save_pretrained('best_model')
print('Model saved to ./best_model/')

final_df.to_csv('model_results.csv', index=False)
print('Results saved to model_results.csv')

import glob as _glob
artifacts = sorted(_glob.glob('*.png') + _glob.glob('*.csv') + _glob.glob('*.html'))
print('\nSaved artifacts:')
for f in artifacts: print(' ', f)